[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadErlanggaKurniawan/ets-ml2-kelompok/blob/main/notebooks/eksperimen_rijal_rl.ipynb)

# Eksperimen Reinforcement Learning: Seberapa Peka Agen terhadap Reward?
**Dikerjakan oleh:** Rijal Apuila Khufi · lanjutan dari notebook utama `ETS_ML2_Kesegaran_Ayam.ipynb` (slide 8–9)

Notebook ini menguji **agen reinforcement learning** lebih jauh. Data dan fiturnya sama persis dengan notebook utama, jadi angka
awalnya harus sama dengan slide.

## Cara mengerjakan
1. Jalankan sel dari atas ke bawah (**Shift + Enter**), atau pilih Runtime → Run all.
2. Di sel bertanda **✏️ UBAH DI SINI**, coba ganti angkanya, jalankan lagi, dan perhatikan hasilnya.
3. Di sel bertanda **✍️ TULIS DI SINI**, tulis pengamatanmu dengan kata-katamu sendiri (klik dua kali sel
   itu untuk mengedit).
4. Setiap selesai satu langkah, simpan ke GitHub: **File → Save a copy in GitHub**. Pilih repo
   `MuhammadErlanggaKurniawan/ets-ml2-kelompok`, branch `main`, path `notebooks/eksperimen_rijal_rl.ipynb`, lalu tulis pesan commit singkat (contoh di tiap langkah).
   Satu kali simpan = satu commit atas nama akunmu.

---
## Langkah 0 — Siapkan data (sama dengan notebook utama)
Sel ini mengunduh data, membuat label, merangkum per 5 menit, dan membuat 19 fitur. Cukup dijalankan, tidak perlu
diubah. Butuh sekitar 30 detik.

In [ ]:
import hashlib, urllib.request, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings("ignore")
SEED = 42
CLASSES = ["Fresh", "Semi-fresh", "Spoiled"]
URL = ("https://raw.githubusercontent.com/talhaanwarch/chicken_assessment_e-nose/"
       "e7e3ca686bce0ef3e5c59f9833c85a3190c36e6e/complete_data.csv")
SHA256 = "7c9920fcfe523af1ee2855d53f13334ddba93db5135d7868b9f63c5cd488d953"
path = Path("data/raw/complete_data.csv")
if not path.exists():
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, path)
assert hashlib.sha256(path.read_bytes()).hexdigest() == SHA256, "Isi file berbeda dari sumber asli"

SENSOR_NAMES = {"s1": "MQ-135", "s2": "MQ-2", "s3": "MQ-3", "s4": "MQ-6", "s5": "MQ-7", "s6": "MQ-9",
                "s8": "Kelembapan"}
raw = pd.read_csv(path).rename(columns={**SENSOR_NAMES, "hour": "jam", "group": "sesi"})
MQ = ["MQ-135", "MQ-2", "MQ-3", "MQ-6", "MQ-7", "MQ-9"]
SENSORS = MQ + ["Kelembapan"]
long_session = raw.groupby("sesi")["jam"].transform("max") > 15
raw["label"] = np.select([raw.jam <= np.where(long_session, 8, 5), raw.jam <= np.where(long_session, 16, 10)],
                         [0, 1], 2)


def make_windows(readings):
    r = readings.copy()
    r["urutan"] = r.groupby(["sesi", "jam"]).cumcount()
    r["n"] = r.groupby(["sesi", "jam"])["jam"].transform("size")
    r["jendela"] = r.urutan * 12 // r.n
    return r.groupby(["sesi", "jam", "jendela"]).agg(**{c: (c, "mean") for c in SENSORS},
                                                    label=("label", "first")).reset_index()


def make_features(win):
    level = np.log1p(win[MQ].clip(lower=0))
    reference = level.where(win.jam == 1).groupby(win.sesi).transform("mean")
    keep = (win.jam >= 2).to_numpy()
    rows = win[keep].reset_index(drop=True)
    level, reference = level[keep].reset_index(drop=True), reference[keep].reset_index(drop=True)
    gas = rows[MQ].clip(lower=0)
    features = pd.concat([level.add_prefix("level_"), gas.div(gas.sum(axis=1) + 1, axis=0).add_prefix("porsi_"),
                          (level - reference).add_prefix("naik_"), rows[["Kelembapan"]]], axis=1)
    return features, rows


features, data = make_features(make_windows(raw))
X, y, groups = features.to_numpy(), data.label.to_numpy(), data.sesi.to_numpy()


def loso(make_model, X_=None):
    # every window is predicted by a model that never saw its session
    X_ = X if X_ is None else X_
    pred, proba = np.empty_like(y), np.zeros((len(y), 3))
    for train, test in LeaveOneGroupOut().split(X_, y, groups):
        model = make_model().fit(X_[train], y[train])
        pred[test], proba[test] = model.predict(X_[test]), model.predict_proba(X_[test])
    return pred, proba


def score(pred):
    return {"akurasi": round(accuracy_score(y, pred), 3), "macro_f1": round(f1_score(y, pred, average="macro"), 3)}


make_rf = lambda: RandomForestClassifier(n_estimators=300, max_depth=6, random_state=SEED, n_jobs=-1)
print(f"Data siap: {len(data)} jendela, {X.shape[1]} fitur, {len(np.unique(groups))} sesi.")

---
## Langkah 1 — Siapkan tebakan Random Forest dan agen
Agen melihat tebakan Random Forest (yang tidak pernah melihat sesi itu) dan memilih **Jual / Cek manual / Buang**.
Fungsi `jalankan_agen` di bawah sama persis dengan notebook utama, tapi reward dan pengaturannya bisa diubah.

In [ ]:
_, proba_rf = loso(make_rf)
predicted = proba_rf.argmax(axis=1)
layak = y <= 1                                   # Fresh or Semi-fresh = still fine to sell
ACTIONS = ["Jual", "Cek manual", "Buang"]
STATE_NAMES = [f"{c} ({'ragu' if u else 'yakin'})" for c in CLASSES for u in (False, True)]
# same fixed session split as the main notebook
RL_TEST_SESSIONS = [[0, 10, 11], [2, 8, 13], [1, 6, 9], [4, 7, 14], [3, 5, 12]]
FOLDS = [(np.flatnonzero(~np.isin(groups, t)), np.flatnonzero(np.isin(groups, t))) for t in RL_TEST_SESSIONS]


def jalankan_agen(jual_layak=1.0, jual_busuk=-5.0, cek=0.5, buang_layak=-1.0, buang_busuk=1.0,
                  ambang_yakin=0.7, epsilon=0.1, iterasi=3000, seeds=5):
    state = predicted * 2 + (proba_rf.max(axis=1) < ambang_yakin)
    reward = np.column_stack([np.where(layak, jual_layak, jual_busuk), np.full(len(y), cek),
                              np.where(layak, buang_layak, buang_busuk)])
    curves, policies = [], []
    for train_idx, test_idx in FOLDS:
        for seed in range(seeds):
            rng = np.random.default_rng(seed)
            Q, N, curve = np.zeros((6, 3)), np.zeros((6, 3)), []
            for t in range(1, iterasi + 1):
                i = rng.choice(train_idx)
                s = state[i]
                a = rng.integers(3) if rng.random() < epsilon else int(np.argmax(Q[s]))
                N[s, a] += 1
                Q[s, a] += (reward[i, a] - Q[s, a]) / N[s, a]
                if t == 1 or t % 100 == 0:
                    curve.append(reward[test_idx, Q.argmax(axis=1)[state[test_idx]]].mean())
            curves.append(curve)
            policies.append(Q.argmax(axis=1))
    policy = [int(np.bincount([p[s] for p in policies], minlength=3).argmax()) for s in range(6)]
    action = np.array(policy)[state]
    return {"reward_akhir": round(float(np.mean(curves, axis=0)[-1]), 3),
            "aturan": dict(zip(STATE_NAMES, [ACTIONS[a] for a in policy])),
            "busuk_terjual": int(((action == 0) & ~layak).sum()),
            "layak_terbuang": int(((action == 2) & layak).sum()),
            "persen_cek": round(100 * float((action == 1).mean()), 1),
            "kurva": np.mean(curves, axis=0)}


def tabel_aturan(hasil_per_setting):
    return pd.DataFrame({k: v["aturan"] for k, v in hasil_per_setting.items()})


def tabel_dampak(hasil_per_setting):
    cols = ["reward_akhir", "busuk_terjual", "layak_terbuang", "persen_cek"]
    return pd.DataFrame({k: {c: v[c] for c in cols} for k, v in hasil_per_setting.items()}).T


print("Siap.")

---
## Langkah 2 — Ulangi hasil di slide
Dengan reward asli, hasilnya harus: **reward akhir 0,61**, aturan Fresh → Jual, Semi-fresh → Cek manual,
Spoiled yakin → Buang, Spoiled ragu → Cek manual, dan **0 ayam busuk terjual**.

💾 Pesan commit contoh: `ulangi hasil agen RL`

In [ ]:
asli = jalankan_agen()
print("Reward akhir:", asli["reward_akhir"], "| busuk terjual:", asli["busuk_terjual"],
      "| cek manual:", asli["persen_cek"], "%")
display(pd.Series(asli["aturan"], name="aksi").to_frame())

### ✍️ TULIS DI SINI
Apakah hasilnya sama dengan slide 8 dan 9?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 3 — Kalau denda menjual ayam busuk diubah
Di slide, menjual ayam busuk diberi **−5**. Bagaimana kalau dendanya lebih ringan atau lebih berat?

💾 Pesan commit contoh: `uji denda jual ayam busuk`

In [ ]:
DENDA = [-2, -5, -10]   # ✏️ UBAH DI SINI

hasil_denda = {f"jual busuk {d}": jalankan_agen(jual_busuk=d) for d in DENDA}
display(tabel_aturan(hasil_denda))
display(tabel_dampak(hasil_denda))

### ✍️ TULIS DI SINI
Aturan mana yang berubah saat denda diubah? Berapa ayam busuk yang terjual di tiap setting?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 4 — Kalau biaya cek manual diubah
Di slide, cek manual diberi **+0,5** (aman, tapi ada biaya). Makin kecil angkanya, makin mahal cek manual.

💾 Pesan commit contoh: `uji biaya cek manual`

In [ ]:
NILAI_CEK = [0.2, 0.5, 0.8]   # ✏️ UBAH DI SINI

hasil_cek = {f"cek {c}": jalankan_agen(cek=c) for c in NILAI_CEK}
display(tabel_aturan(hasil_cek))
display(tabel_dampak(hasil_cek))

### ✍️ TULIS DI SINI
Kalau cek manual makin mahal, apakah agen makin jarang mengecek? Apa risikonya?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 5 — Kalau ambang "yakin" diubah
Model dianggap **yakin** kalau peluang tebakannya ≥ ambang. Di slide ambangnya 0,7.

💾 Pesan commit contoh: `uji ambang yakin`

In [ ]:
AMBANG = [0.6, 0.7, 0.8]   # ✏️ UBAH DI SINI

hasil_ambang = {f"ambang {a}": jalankan_agen(ambang_yakin=a) for a in AMBANG}
display(tabel_aturan(hasil_ambang))
display(tabel_dampak(hasil_ambang))

### ✍️ TULIS DI SINI
Apa efek menaikkan atau menurunkan ambang terhadap jumlah cek manual dan keamanan?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 6 — Kalau ε (seberapa sering agen coba-coba) diubah
ε = 0,1 artinya 10% aksi dipilih acak untuk eksplorasi.

💾 Pesan commit contoh: `uji epsilon agen`

In [ ]:
EPSILON = [0.01, 0.1, 0.3]   # ✏️ UBAH DI SINI

fig, ax = plt.subplots(figsize=(8, 4))
langkah = np.r_[1, np.arange(100, 3001, 100)]
for e in EPSILON:
    h = jalankan_agen(epsilon=e)
    ax.plot(langkah, h["kurva"], marker="o", ms=3, label=f"ε = {e} (akhir {h['reward_akhir']})")
ax.set(xlabel="iterasi belajar agen", ylabel="rata-rata reward di sesi uji", title="Pengaruh ε terhadap belajar agen")
ax.legend()
plt.show()

### ✍️ TULIS DI SINI
ε mana yang belajarnya paling cepat? Mana yang hasil akhirnya paling baik?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 7 — Kesimpulan
💾 Pesan commit contoh: `tulis kesimpulan eksperimen RL`

### ✍️ TULIS DI SINI
Tulis 3–4 kalimat: seberapa peka agen terhadap angka reward, setting mana yang menurutmu paling masuk akal untuk toko sungguhan, dan apa batasannya.

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*